# 23.3 Cooperative game theory

When players can form coalitions, the question changes from which action to choose to how value should be divided. This lesson uses set functions, permutations, and coalition stability to connect Shapley attribution with the core. Save a copy to Drive to edit.

In [ ]:
import itertools
import math
import random

import matplotlib.pyplot as plt
import numpy as np

SEED = 233
random.seed(SEED)
np.random.seed(SEED)

## The concept, built once: Shapley values

For a cooperative game $v:2^N\to\mathbb{R}$, the Shapley value is

$$\phi_i=\sum_{S\subseteq N\setminus\{i\}}\frac{|S|!(n-|S|-1)!}{n!}\big(v(S\cup\{i\})-v(S)\big).$$

The lesson game has $v(A)=1$, $v(B)=1$, $v(C)=0$, $v(AB)=4$, $v(AC)=2$, $v(BC)=2$, and $v(ABC)=6$, producing $(2.5,2.5,1.0)$.

In [ ]:
def key(coalition):
    return frozenset(coalition)


def value_lookup(v, coalition):
    return float(v.get(key(coalition), 0.0))


def shapley_values(v, players):
    players = list(players)
    n = len(players)
    phi = {player: 0.0 for player in players}

    for player in players:
        others = [other for other in players if other != player]
        for size in range(len(others) + 1):
            for coalition in itertools.combinations(others, size):
                weight = math.factorial(size) * math.factorial(n - size - 1) / math.factorial(n)
                before = value_lookup(v, coalition)
                after = value_lookup(v, tuple(coalition) + (player,))
                phi[player] += weight * (after - before)

    return phi


lesson_players = ["A", "B", "C"]
lesson_v = {
    key(()): 0,
    key(("A",)): 1,
    key(("B",)): 1,
    key(("C",)): 0,
    key(("A", "B")): 4,
    key(("A", "C")): 2,
    key(("B", "C")): 2,
    key(("A", "B", "C")): 6,
}
lesson_phi = shapley_values(lesson_v, lesson_players)
print(lesson_phi)
assert round(lesson_phi["A"], 6) == 2.5
assert round(lesson_phi["B"], 6) == 2.5
assert round(lesson_phi["C"], 6) == 1.0
assert round(sum(lesson_phi.values()), 6) == 6.0

Now separate fairness from stability. The core check asks whether each coalition receives at least $v(S)$. In the lesson allocation, coalition $AB$ receives $2.5+2.5=5.0\ge4$.

In [ ]:
def core_violations(allocation, v, players):
    players = list(players)
    deficits = []
    for size in range(1, len(players) + 1):
        for coalition in itertools.combinations(players, size):
            allocated = sum(allocation[player] for player in coalition)
            required = value_lookup(v, coalition)
            deficit = required - allocated
            if deficit > 1e-8:
                deficits.append((coalition, deficit, required, allocated))
    return deficits


lesson_violations = core_violations(lesson_phi, lesson_v, lesson_players)
lesson_ab_allocation = lesson_phi["A"] + lesson_phi["B"]
print("AB allocation", lesson_ab_allocation, "violations", lesson_violations)
assert round(lesson_ab_allocation, 6) == 5.0
assert lesson_ab_allocation >= 4.0
assert not lesson_violations

## The dataset ladder: D1 to D5 coalition profiles

The ladder grows from the 3-player lesson characteristic function to larger synergy, redundancy, sampled coalition, and empty-core-like games.

In [ ]:
def all_coalitions(players):
    coalitions = []
    for size in range(len(players) + 1):
        coalitions.extend(itertools.combinations(players, size))
    return coalitions


def additive_game(players, weights):
    v = {}
    for coalition in all_coalitions(players):
        v[key(coalition)] = sum(weights[player] for player in coalition)
    return v


def build_cooperative_ladder():
    ladder = []
    ladder.append({"name": "D1 lesson game", "players": ["A", "B", "C"], "v": lesson_v})
    players2 = ["A", "B", "C", "D"]
    base2 = additive_game(players2, {"A": 2, "B": 2, "C": 1, "D": 1})
    ladder.append({"name": "D2 additive dummy", "players": players2, "v": base2})
    players3 = ["A", "B", "C", "D", "E"]
    v3 = {}
    for coalition in all_coalitions(players3):
        coalition_set = set(coalition)
        value = len(coalition_set)
        if {"A", "B"}.issubset(coalition_set):
            value += 3
        if {"C", "D", "E"}.issubset(coalition_set):
            value += 4
        v3[key(coalition)] = value
    ladder.append({"name": "D3 synergy and redundancy", "players": players3, "v": v3})
    players4 = ["P0", "P1", "P2", "P3", "P4", "P5"]
    weights4 = {player: index + 1 for index, player in enumerate(players4)}
    v4 = additive_game(players4, weights4)
    for coalition in all_coalitions(players4):
        if len(coalition) >= 3:
            v4[key(coalition)] += 2
    ladder.append({"name": "D4 six-player sampled-style", "players": players4, "v": v4})
    players5 = ["A", "B", "C", "D", "E"]
    v5 = {}
    for coalition in all_coalitions(players5):
        v5[key(coalition)] = 1.0 if len(coalition) >= 3 else 0.0
    ladder.append({"name": "D5 majority game with complaints", "players": players5, "v": v5})
    return ladder


coop_ladder = build_cooperative_ladder()
for rung in coop_ladder:
    grand = value_lookup(rung["v"], rung["players"])
    print(rung["name"], "players", len(rung["players"]), "grand value", grand)

## Run the SAME method across D1-D5

Compute Shapley values and coalition-deficit metrics for every characteristic-function rung.

In [ ]:
coop_results = []
for rung in coop_ladder:
    phi = shapley_values(rung["v"], rung["players"])
    grand = value_lookup(rung["v"], rung["players"])
    efficiency_gap = abs(sum(phi.values()) - grand)
    violations = core_violations(phi, rung["v"], rung["players"])
    worst_deficit = max([item[1] for item in violations], default=0.0)
    coop_results.append({
        "name": rung["name"],
        "players": len(rung["players"]),
        "phi": phi,
        "gap": efficiency_gap,
        "worst_deficit": worst_deficit,
        "violations": violations,
    })

for row in coop_results:
    print(row["name"], "gap", round(row["gap"], 6), "worst deficit", round(row["worst_deficit"], 6), "phi", row["phi"])

## Results visualization

The closing figure has Shapley bar panels plus deficit and efficiency curves over player count.

In [ ]:
fig, axes = plt.subplots(2, len(coop_results), figsize=(18, 7))

for axis, row in zip(axes[0], coop_results):
    names = list(row["phi"].keys())
    values = [row["phi"][name] for name in names]
    axis.bar(names, values)
    axis.set_title(row["name"].split()[0] + " Shapley")
    axis.tick_params(axis="x", rotation=45)

player_counts = [row["players"] for row in coop_results]
deficits = [row["worst_deficit"] for row in coop_results]
gaps = [row["gap"] for row in coop_results]
axes[1, 0].plot(player_counts, deficits, marker="o", label="worst coalition deficit")
axes[1, 0].set_xlabel("players")
axes[1, 0].set_ylabel("deficit")
axes[1, 0].legend()
axes[1, 1].plot(player_counts, gaps, marker="s", color="tab:green", label="efficiency gap")
axes[1, 1].set_xlabel("players")
axes[1, 1].set_ylabel("gap")
axes[1, 1].legend()
for axis in axes[1, 2:]:
    axis.axis("off")
plt.tight_layout()
plt.show()

## Pitfall on D5: Shapley is not the core

The Shapley allocation is efficient, but a coalition may still complain. D5 reproduces an efficient allocation with a positive blocking-coalition deficit, then fixes the analysis by reporting core deficits separately.

In [ ]:
d5 = coop_ladder[-1]
d5_phi = shapley_values(d5["v"], d5["players"])
d5_violations = core_violations(d5_phi, d5["v"], d5["players"])
wrong_claim = abs(sum(d5_phi.values()) - value_lookup(d5["v"], d5["players"])) < 1e-8
worst = max(d5_violations, key=lambda item: item[1])
print("wrong efficient-implies-core claim", wrong_claim)
print("blocking coalition", worst[0], "requires", worst[2], "gets", round(worst[3], 6), "deficit", round(worst[1], 6))
assert wrong_claim
assert worst[1] > 0

## Evaluate it + Practice

- Metric: efficiency gap and worst coalition deficit; compare with an equal-split baseline.
- Sanity check: Shapley values must sum to $v(N)$.
- Ablation: remove the synergy term in D3 and watch the marginal contributions flatten.
- Failure signal: a positive coalition deficit means Shapley is fair by axioms but not core-stable.

Practice 1: Add a dummy player to D1 that contributes exactly 1 to every coalition and verify its Shapley value.

Practice 2: Replace D5 with a 4-player majority game and find the worst blocking coalition.